# Three-Dimensional Circular Masked Stromatolite Growth Model

This notebook implements a three-dimensional circular stromatolite model by applying a circular active mask to the regular x-y grid used by the rectangular model. Each active grid cell retains the same interpretable biology as the one-dimensional, two-dimensional, and rectangular three-dimensional models, while inactive cells outside the footprint remain excluded from growth, sediment trapping, diagnostics, and visualisation.

The model remains height-field based rather than voxel based. Each timestep records a masked raster of newly added material across the active circular surface, so the final output can be inspected as a three-dimensional topography, a plan-view map, and selected cross-sectional slices.


## Model Rules

- The surface is a regular x-y grid, but only cells inside a circular footprint are active stromatolite columns.
- The circular mask is centred on the grid and controlled by a configurable footprint radius.
- Active cells keep their own height, biomass, EPS, trapped sediment, and local lamina identifier.
- Inactive cells outside the mask remain NaN or marked inactive and do not receive growth, sediment, smoothing, burial, or diagnostic weighting.
- Incident light and temperature follow the same seasonal forcing functions as the 1-D, 2-D, and rectangular 3-D models.
- Water depth is computed locally from the water level and each active cell height, so elevated cells receive more light.
- Photosynthetic biomass growth follows a Monod-style light response multiplied by temperature suitability.
- Biomass produces EPS, and EPS controls the fraction of supplied sediment retained by each active cell.
- Sediment supply has shared seasonal and storm forcing plus local spatial weather noise across the x-y grid.
- Burial is triggered locally when either one timestep traps enough sediment or cumulative active sediment exceeds the burial threshold.
- Buried cells are reseeded with a new active mat while unburied neighbours continue growing in their existing lamina.
- A weak two-dimensional smoothing step operates only across active neighbours and uses a no-flux edge at the circular boundary.


In [ ]:
%run common.ipynb

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, fields
from pathlib import Path
import math

import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
CONFIG_DIR = DATA_DIR / "config"
OUTPUT_DIR = DATA_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_DIR = PROJECT_ROOT / "diagrams"
BIOLOGY_CONFIG_FILE = CONFIG_DIR / "biology.config"
GEOMETRY_CONFIG_FILE = CONFIG_DIR / "3d-circular.config"

In [ ]:
@dataclass
class ModelParameters3D(ModelParameters1D):
    """Container for the 3-D circular masked model parameters.

    :param nx: Number of grid cells in the x direction of the rectangular computational grid.
    :param ny: Number of grid cells in the y direction of the rectangular computational grid.
    :param dx_mm: Horizontal spacing between neighbouring grid cells in both x and y directions.
    :param footprint_radius_mm: Optional radius of the active circular footprint; None uses the largest centred circle that fits inside the grid.
    :param lateral_smoothing_rate: Fractional strength of nearest-neighbour surface smoothing applied within the circular footprint each timestep.
    :param spatial_sediment_noise: Relative random spatial variation applied to sediment supply across the active circular footprint.
    :param snapshot_count: Number of evenly spaced surface snapshots retained for visualisation.
    :param slice_y_index: Optional y-index used for the final cross-section slice; None selects the centre row.
    :param surface_render_z_padding_fraction: Fractional vertical padding added around the rendered final surface height range.
    """

    nx: int
    ny: int
    dx_mm: float
    footprint_radius_mm: float | None
    lateral_smoothing_rate: float
    spatial_sediment_noise: float
    snapshot_count: int
    slice_y_index: int | None
    surface_render_z_padding_fraction: float


In [ ]:
def load_3d_parameters(biology_path: Path, geometry_path: Path) -> ModelParameters3D:
    """Load shared biology and 3-D circular geometry configuration.

    :param biology_path: Path to the shared biology JSON config file.
    :param geometry_path: Path to the 3-D circular geometry JSON config file.
    :return: A populated 3-D parameters instance.
    """
    raw_values = load_parameter_values(biology_path)
    raw_values.update(load_parameter_values(geometry_path))

    integer_parameters = {"time_steps", "random_seed", "nx", "ny", "snapshot_count"}
    for name in integer_parameters & raw_values.keys():
        raw_values[name] = int(raw_values[name])
    nullable_float_parameters = {"footprint_radius_mm"}
    for name in nullable_float_parameters & raw_values.keys():
        if raw_values[name] is not None:
            raw_values[name] = float(raw_values[name])
    if raw_values.get("slice_y_index") is not None:
        raw_values["slice_y_index"] = int(raw_values["slice_y_index"])

    allowed_names = {field.name for field in fields(ModelParameters3D)}
    filtered_values = {
        name: value
        for name, value in raw_values.items()
        if name in allowed_names
    }
    return ModelParameters3D(**filtered_values)


In [ ]:
def circular_active_mask(params: ModelParameters3D) -> np.ndarray:
    """Create a centred circular Boolean mask across the x-y grid.

    :param params: Model parameters controlling grid size, spacing, and optional radius.
    :return: Boolean raster where True cells are active stromatolite columns.
    """

    # Safe to use dx_mm for both X and Y when masking as we currently use square cells
    x_mm = np.arange(params.nx, dtype=float) * params.dx_mm
    y_mm = np.arange(params.ny, dtype=float) * params.dx_mm

    x_grid, y_grid = np.meshgrid(x_mm, y_mm)
    centre_x = 0.5 * (x_mm.min() + x_mm.max())
    centre_y = 0.5 * (y_mm.min() + y_mm.max())
    default_radius = 0.5 * min(x_mm.max() - x_mm.min(), y_mm.max() - y_mm.min())
    radius = default_radius if params.footprint_radius_mm is None else params.footprint_radius_mm
    return (x_grid - centre_x) ** 2 + (y_grid - centre_y) ** 2 <= radius ** 2




In [ ]:
def initialise_3d_state(params: ModelParameters3D) -> dict:
    """Create the initial flat active surface state for the circular domain.

    :param params: Model parameters controlling grid size, spacing, and circular footprint.
    :return: Mutable model state arrays.
    """
    x_mm = np.arange(params.nx, dtype=float) * params.dx_mm
    y_mm = np.arange(params.ny, dtype=float) * params.dx_mm
    active_mask = circular_active_mask(params)
    height = np.full((params.ny, params.nx), np.nan, dtype=float)
    height[active_mask] = params.initial_mat_thickness_mm

    biomass = np.full((params.ny, params.nx), np.nan, dtype=float)
    biomass[active_mask] = params.new_layer_biomass_seed
    eps = np.full((params.ny, params.nx), np.nan, dtype=float)
    eps[active_mask] = params.new_layer_eps_seed
    active_sediment = np.full((params.ny, params.nx), np.nan, dtype=float)
    active_sediment[active_mask] = 0.0
    layer_id = np.full((params.ny, params.nx), -1, dtype=int)
    layer_id[active_mask] = 0

    return {
        "x_mm": x_mm,
        "y_mm": y_mm,
        "active_mask": active_mask,
        "height_mm": height,
        "biomass": biomass,
        "eps": eps,
        "active_sediment_mm": active_sediment,
        "layer_id": layer_id,
    }


In [ ]:
def update_active_surface(state: dict, params: ModelParameters3D, step: int, rng: np.random.Generator) -> dict:
    """Advance all exposed microbial mats by one timestep.

    :param state: Mutable 3-D circular masked model state arrays.
    :param params: Model parameters controlling growth, sediment trapping, and burial.
    :param step: Current simulation step.
    :param rng: NumPy random generator used for sediment-supply variation.
    :return: Dictionary of timestep diagnostics and masked raster layers.
    """
    active_mask = state["active_mask"]
    cell_area_mm2 = params.dx_mm * params.dx_mm

    # Place this timestep within the annual cycle so light, temperature, and sediment
    # forcing represent the current season experienced by the microbial mat.
    current_day = day_of_year(step, params.dt_days, params.days_per_year)
    light_forcing = light_at_surface(current_day, state["height_mm"], params)
    temperature_today = seasonal_temperature(current_day, params)
    temp_factor = temperature_response(
        temperature_today,
        params.optimal_temperature_c,
        params.temperature_width_c,
    )
    local_light = np.where(active_mask, light_forcing["light_at_mat"], np.nan)

    # Convert local mat light and temperature suitability into biological production.
    # Higher surface cells have shallower water cover, so the same biology can diverge
    # spatially as relief develops within the circular footprint.
    growth_rate = monod_growth_rate(local_light, params.max_growth_rate_per_day, params.half_saturation_light) * temp_factor
    eps_production_rate = params.eps_production_rate_per_day * temp_factor
    biomass_change = (growth_rate - params.mortality_rate_per_day) * state["biomass"] * params.dt_days
    eps_change = (eps_production_rate * state["biomass"] - params.eps_decay_rate_per_day * state["eps"]) * params.dt_days

    # Deliver sediment from the water column, then let EPS decide how much is retained
    # on each active surface cell rather than passing over the mat.
    sediment_forcing = spatial_sediment_supply(current_day, params, rng)
    supplied_sediment = np.where(active_mask, sediment_forcing["local_sediment_supply_mm_per_day"] * params.dt_days, np.nan)
    retained_fraction = trapping_fraction(state["eps"], params.trapping_efficiency, params.eps_half_saturation)
    trapped_sediment = supplied_sediment * retained_fraction

    # Translate newly produced biomass, EPS, and trapped sediment into vertical accretion
    # of the stromatolite surface for this day.
    positive_biomass_change = np.maximum(0.0, biomass_change)
    positive_eps_change = np.maximum(0.0, eps_change)
    biological_increment = (
        positive_biomass_change * params.biomass_to_thickness
        + positive_eps_change * params.eps_to_thickness
    )
    sediment_increment = trapped_sediment * params.sediment_to_thickness
    total_increment = biological_increment + sediment_increment

    state["biomass"][active_mask] = np.maximum(0.0, state["biomass"][active_mask] + biomass_change[active_mask])
    state["eps"][active_mask] = np.maximum(0.0, state["eps"][active_mask] + eps_change[active_mask])
    state["active_sediment_mm"][active_mask] += trapped_sediment[active_mask]
    state["height_mm"][active_mask] += total_increment[active_mask]

    # Identify active cells where sediment has locally covered the mat. Those cells
    # close one lamina and start a fresh microbial surface, while unburied neighbours
    # continue growing in their existing lamina.
    buried = active_mask & (
        (trapped_sediment >= params.rapid_burial_sediment_threshold_mm)
        | (state["active_sediment_mm"] >= params.burial_sediment_threshold_mm)
    )
    if buried.any():
        state["layer_id"][buried] += 1
        state["active_sediment_mm"][buried] = 0.0
        state["biomass"][buried] = params.new_layer_biomass_seed
        state["eps"][buried] = params.new_layer_eps_seed
        state["height_mm"][buried] += params.initial_mat_thickness_mm
        total_increment = total_increment.copy()
        biological_increment = biological_increment.copy()
        total_increment[buried] += params.initial_mat_thickness_mm
        biological_increment[buried] += params.initial_mat_thickness_mm

    # Let active neighbouring cells share a little relief after growth and burial,
    # while inactive cells outside the footprint remain empty.
    state["height_mm"] = apply_surface_smoothing(state["height_mm"], active_mask, params)

    # Record the composition of the material deposited during this timestep, as opposed
    # to the current active surface state.
    sediment_fraction = np.divide(
        sediment_increment,
        total_increment,
        out=np.full_like(total_increment, np.nan),
        where=active_mask & (total_increment > 0),
    )
    biological_signal = np.divide(
        biological_increment,
        total_increment,
        out=np.full_like(total_increment, np.nan),
        where=active_mask & (total_increment > 0),
    )
    deposition_increment = np.where(active_mask, total_increment, np.nan)
    sediment_increment = np.where(active_mask, sediment_increment, np.nan)
    water_depth = np.where(active_mask, light_forcing["water_depth_above_mat_mm"], np.nan)

    active_height = active_values(state["height_mm"], active_mask)
    active_biomass = active_values(state["biomass"], active_mask)
    active_eps = active_values(state["eps"], active_mask)
    active_sediment = active_values(state["active_sediment_mm"], active_mask)
    active_light = active_values(local_light, active_mask)
    active_water_depth = active_values(water_depth, active_mask)
    active_supplied = active_values(supplied_sediment, active_mask)
    active_trapped = active_values(trapped_sediment, active_mask)
    active_cell_count = int(np.count_nonzero(active_mask))

    return {
        "step": step + 1,
        "day_of_year": current_day,
        "incident_light": light_forcing["incident_light"],
        "temperature_c": temperature_today,
        "temperature_factor": temp_factor,
        "sediment_seasonal_multiplier": sediment_forcing["sediment_seasonal_multiplier"],
        "sediment_weather_multiplier": sediment_forcing["sediment_weather_multiplier"],
        "storm_sediment_mm_per_day": sediment_forcing["storm_sediment_mm_per_day"],
        "storm_today": sediment_forcing["storm_today"],
        "sediment_supply_mm_per_day": sediment_forcing["sediment_supply_mm_per_day"],
        "active_cell_count": active_cell_count,
        "active_area_mm2": float(active_cell_count * cell_area_mm2),
        "mean_height_mm": float(np.mean(active_height)),
        "max_height_mm": float(np.max(active_height)),
        "min_height_mm": float(np.min(active_height)),
        "relief_mm": float(np.max(active_height) - np.min(active_height)),
        "height_variance_mm2": float(np.var(active_height)),
        "surface_roughness_mm": float(np.std(active_height)),
        "mean_biomass": float(np.mean(active_biomass)),
        "total_biomass": float(np.sum(active_biomass)),
        "mean_eps": float(np.mean(active_eps)),
        "mean_active_sediment_mm": float(np.mean(active_sediment)),
        "total_active_sediment_mm": float(np.sum(active_sediment)),
        "mean_light_at_mat": float(np.mean(active_light)),
        "min_light_at_mat": float(np.min(active_light)),
        "max_light_at_mat": float(np.max(active_light)),
        "mean_water_depth_above_mat_mm": float(np.mean(active_water_depth)),
        "mean_supplied_sediment_mm": float(np.mean(active_supplied)),
        "mean_trapped_sediment_mm": float(np.mean(active_trapped)),
        "total_deposition_volume_mm3": float(np.nansum(deposition_increment) * cell_area_mm2),
        "total_sediment_volume_mm3": float(np.nansum(sediment_increment) * cell_area_mm2),
        "burial_count": int(np.count_nonzero(buried)),
        "exposed_surface_fraction": float(np.count_nonzero(active_mask & ~buried) / active_cell_count),
        "surface_height_mm": state["height_mm"].copy(),
        "biomass_grid": state["biomass"].copy(),
        "eps_grid": state["eps"].copy(),
        "active_sediment_grid_mm": state["active_sediment_mm"].copy(),
        "light_grid": local_light.copy(),
        "deposition_increment_mm": deposition_increment.copy(),
        "sediment_fraction": sediment_fraction.copy(),
        "biological_signal": biological_signal.copy(),
        "buried": buried.copy(),
        "layer_id": state["layer_id"].copy(),
    }


In [ ]:
def summarise_initial_state(state: dict, params: ModelParameters3D) -> dict:
    """Summarise the initial circular surface before any timestep update.

    :param state: Mutable 3-D model state arrays.
    :return: Summary metrics for the initial state.
    """
    active_mask = state["active_mask"]
    active_height = active_values(state["height_mm"], active_mask)
    active_biomass = active_values(state["biomass"], active_mask)
    active_eps = active_values(state["eps"], active_mask)
    active_sediment = active_values(state["active_sediment_mm"], active_mask)
    active_cell_count = int(np.count_nonzero(active_mask))

    return {
        "step": 0,
        "day_of_year": 0.0,
        "incident_light": np.nan,
        "temperature_c": np.nan,
        "temperature_factor": np.nan,
        "sediment_seasonal_multiplier": np.nan,
        "sediment_weather_multiplier": np.nan,
        "storm_sediment_mm_per_day": 0.0,
        "storm_today": False,
        "sediment_supply_mm_per_day": np.nan,
        "active_cell_count": active_cell_count,
        "active_area_mm2": float(active_cell_count * params.dx_mm * params.dx_mm),
        "mean_height_mm": float(np.mean(active_height)),
        "max_height_mm": float(np.max(active_height)),
        "min_height_mm": float(np.min(active_height)),
        "relief_mm": float(np.max(active_height) - np.min(active_height)),
        "height_variance_mm2": float(np.var(active_height)),
        "surface_roughness_mm": float(np.std(active_height)),
        "mean_biomass": float(np.mean(active_biomass)),
        "total_biomass": float(np.sum(active_biomass)),
        "mean_eps": float(np.mean(active_eps)),
        "mean_active_sediment_mm": float(np.mean(active_sediment)),
        "total_active_sediment_mm": float(np.sum(active_sediment)),
        "mean_light_at_mat": np.nan,
        "min_light_at_mat": np.nan,
        "max_light_at_mat": np.nan,
        "mean_water_depth_above_mat_mm": np.nan,
        "mean_supplied_sediment_mm": 0.0,
        "mean_trapped_sediment_mm": 0.0,
        "total_deposition_volume_mm3": 0.0,
        "total_sediment_volume_mm3": 0.0,
        "burial_count": 0,
        "exposed_surface_fraction": 1.0,
        "surface_height_mm": state["height_mm"].copy(),
        "biomass_grid": state["biomass"].copy(),
        "eps_grid": state["eps"].copy(),
        "active_sediment_grid_mm": state["active_sediment_mm"].copy(),
        "light_grid": np.full_like(state["height_mm"], np.nan),
    }


In [ ]:
def run_3d_model(params: ModelParameters3D) -> tuple[dict, list[dict], dict, list[dict]]:
    """Run the 3-D circular masked stromatolite growth model.

    :param params: Model parameters controlling growth, sediment trapping, burial, and spatial coupling.
    :return: Final state, timestep history, stratigraphic rasters, and selected surface snapshots.
    """
    # Seed one reproducible circular domain and initialise the exposed microbial
    # surface before any seasonal forcing, growth, or burial has occurred.
    rng = np.random.default_rng(params.random_seed)
    state = initialise_3d_state(params)
    history = [summarise_initial_state(state, params)]

    # Select a small set of times that will show how the surface changes from the
    # initial flat mat into the final three-dimensional height field.
    snapshot_steps = set(np.linspace(0, params.time_steps, params.snapshot_count, dtype=int))
    snapshots = []
    if 0 in snapshot_steps:
        snapshots.append({
            "step": 0,
            "surface_height_mm": state["height_mm"].copy(),
            "biomass_grid": state["biomass"].copy(),
            "eps_grid": state["eps"].copy(),
            "active_sediment_grid_mm": state["active_sediment_mm"].copy(),
            "light_grid": np.full((params.ny, params.nx), np.nan),
            "active_mask": state["active_mask"].copy(),
        })

    # These layers become the 3-D stratigraphic archive: each timestep contributes one
    # masked depositional raster across the modeled surface.
    deposition_layers = []
    sediment_fraction_layers = []
    biological_signal_layers = []
    burial_layers = []
    layer_id_layers = []

    for step in range(params.time_steps):
        # Advance the active surface through one day of light exposure, biological
        # growth, sediment trapping, local burial, and weak surface smoothing.
        diagnostics = update_active_surface(state, params, step, rng)
        history.append(diagnostics)

        # Store the new material and burial state as one masked slice of the final
        # laminated archive. Compact dtypes keep the 3-D notebook practical to run.
        deposition_layers.append(diagnostics["deposition_increment_mm"].astype(np.float32))
        sediment_fraction_layers.append(diagnostics["sediment_fraction"].astype(np.float32))
        biological_signal_layers.append(diagnostics["biological_signal"].astype(np.float32))
        burial_layers.append(diagnostics["buried"])
        layer_id_layers.append(diagnostics["layer_id"].astype(np.int16))

        # Preserve selected exposed-surface states so the model can be inspected as a
        # growth sequence, not only as a finished topography.
        if diagnostics["step"] in snapshot_steps:
            snapshots.append({
                "step": diagnostics["step"],
                "surface_height_mm": diagnostics["surface_height_mm"].copy(),
                "biomass_grid": diagnostics["biomass_grid"].copy(),
                "eps_grid": diagnostics["eps_grid"].copy(),
                "active_sediment_grid_mm": diagnostics["active_sediment_grid_mm"].copy(),
                "light_grid": diagnostics["light_grid"].copy(),
                "active_mask": state["active_mask"].copy(),
            })

    # Stack the daily depositional rasters for plotting sediment-rich intervals,
    # biological signal, local burial, and lamina identifiers through time.
    stratigraphy = {
        "deposition_increment_mm": np.stack(deposition_layers),
        "sediment_fraction": np.stack(sediment_fraction_layers),
        "biological_signal": np.stack(biological_signal_layers),
        "burial_map": np.stack(burial_layers),
        "layer_id": np.stack(layer_id_layers),
        "active_mask": state["active_mask"].copy(),
    }
    return state, history, stratigraphy, snapshots


In [ ]:
def plot_growth_summary_3d(history: list[dict], params: ModelParameters3D) -> None:
    """Plot mean height, roughness, burial counts, and material summaries through time.

    :param history: List of timestep summary dictionaries from the model run.
    :param params: Model parameters used to convert steps into days.
    :return: None. A Matplotlib figure is displayed.
    """
    time_days = history_as_array(history, "step") * params.dt_days
    _, axes = plt.subplots(1, 4, figsize=(18, 4), constrained_layout=True)

    axes[0].plot(time_days, history_as_array(history, "mean_height_mm"), color=YLORBR_PALETTE["strong"])
    axes[0].plot(time_days, history_as_array(history, "max_height_mm"), color=YLORBR_PALETTE["dark"], linewidth=1.0, alpha=0.8)
    axes[0].set_title("Surface height")
    axes[0].set_xlabel("Time (days)")
    axes[0].set_ylabel("Height (mm)")

    axes[1].plot(time_days, history_as_array(history, "surface_roughness_mm"), color=YLORBR_PALETTE["deep"], label="roughness")
    axes[1].plot(time_days, history_as_array(history, "relief_mm"), color=YLORBR_PALETTE["mid"], linewidth=1.0, alpha=0.8, label="relief")
    axes[1].set_title("Roughness and relief")
    axes[1].set_xlabel("Time (days)")
    axes[1].set_ylabel("mm")
    axes[1].legend(frameon=False)

    axes[2].plot(time_days, history_as_array(history, "burial_count"), color=YLORBR_PALETTE["deep"], linewidth=1.0)
    axes[2].set_title("Local burial events")
    axes[2].set_xlabel("Time (days)")
    axes[2].set_ylabel("Cells buried")

    axes[3].plot(time_days, history_as_array(history, "mean_biomass"), label="biomass", color=YLORBR_PALETTE["light"])
    axes[3].plot(time_days, history_as_array(history, "mean_eps"), label="EPS", color=YLORBR_PALETTE["mid"])
    axes[3].plot(time_days, history_as_array(history, "mean_active_sediment_mm"), label="active sediment", color=YLORBR_PALETTE["dark"])
    axes[3].set_title("Mean active materials")
    axes[3].set_xlabel("Time (days)")
    axes[3].legend(frameon=False)

    plt.savefig(IMAGE_DIR / "3d-circular-growth-summary.png", dpi=300)
    plt.show()


In [ ]:
def plot_environmental_forcing_3d(history: list[dict], params: ModelParameters3D) -> None:
    """Plot environmental forcing and spatial light range through time.

    :param history: List of timestep summary dictionaries from the model run.
    :param params: Model parameters used to convert steps into days.
    :return: None. A Matplotlib figure is displayed.
    """
    time_days = history_as_array(history, "step") * params.dt_days
    _, axes = plt.subplots(1, 4, figsize=(18, 4), constrained_layout=True)

    axes[0].plot(time_days, history_as_array(history, "incident_light"), label="incident", color=YLORBR_PALETTE["mid"])
    axes[0].plot(time_days, history_as_array(history, "mean_light_at_mat"), label="mean mat", color=YLORBR_PALETTE["dark"])
    axes[0].fill_between(time_days, history_as_array(history, "min_light_at_mat"), history_as_array(history, "max_light_at_mat"), color=YLORBR_PALETTE["pale"], alpha=0.35, linewidth=0.0, label="mat range")
    axes[0].set_title("Spatial mat light")
    axes[0].set_xlabel("Time (days)")
    axes[0].set_ylabel("Relative light")
    axes[0].legend(frameon=False)

    axes[1].plot(time_days, history_as_array(history, "mean_water_depth_above_mat_mm"), color=YLORBR_PALETTE["strong"])
    axes[1].set_title("Mean water above mat")
    axes[1].set_xlabel("Time (days)")
    axes[1].set_ylabel("Depth (mm)")

    axes[2].plot(time_days, history_as_array(history, "temperature_c"), color=YLORBR_PALETTE["deep"])
    axes[2].set_title("Seasonal temperature")
    axes[2].set_xlabel("Time (days)")
    axes[2].set_ylabel("Temperature (deg C)")

    axes[3].plot(time_days, history_as_array(history, "sediment_supply_mm_per_day"), color=YLORBR_PALETTE["dark"], linewidth=1.0)
    storm_mask = history_as_array(history, "storm_today") > 0.0
    if storm_mask.any():
        axes[3].scatter(time_days[storm_mask], history_as_array(history, "sediment_supply_mm_per_day")[storm_mask], color=YLORBR_PALETTE["deep"], s=18, label="storm")
        axes[3].legend(frameon=False)
    axes[3].set_title("Sediment forcing")
    axes[3].set_xlabel("Time (days)")
    axes[3].set_ylabel("mm/day")

    plt.savefig(IMAGE_DIR / "3d-circular-environmental-forcing.png", dpi=300)
    plt.show()


In [ ]:
def plot_surface_snapshots_3d(snapshots: list[dict], params: ModelParameters3D) -> None:
    """Plot selected plan-view surface-height snapshots through time.

    :param snapshots: List of saved surface snapshots.
    :param params: Model parameters controlling grid spacing and time conversion.
    :return: None. A Matplotlib figure is displayed.
    """
    cols = min(3, len(snapshots))
    rows = math.ceil(len(snapshots) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(4.2 * cols, 3.8 * rows), constrained_layout=True)
    axes = np.atleast_1d(axes).ravel()
    extent = [0.0, (params.nx - 1) * params.dx_mm, 0.0, (params.ny - 1) * params.dx_mm]
    vmin = min(float(np.nanmin(snapshot["surface_height_mm"])) for snapshot in snapshots)
    vmax = max(float(np.nanmax(snapshot["surface_height_mm"])) for snapshot in snapshots)
    cmap = circular_plot_cmap()

    for ax, snapshot in zip(axes, snapshots):
        image = ax.imshow(masked_for_plot(snapshot["surface_height_mm"]), origin="lower", extent=extent, cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_title(f"day {snapshot['step'] * params.dt_days:.0f}")
        ax.set_xlabel("x-position (mm)")
        ax.set_ylabel("y-position (mm)")
    for ax in axes[len(snapshots):]:
        ax.set_axis_off()

    fig.colorbar(image, ax=axes[:len(snapshots)], label="Surface height (mm)", shrink=0.85)
    plt.savefig(IMAGE_DIR / "3d-circular-surface-snapshots.png", dpi=300)
    plt.show()


In [ ]:
def plot_final_surface_render(final_state: dict, params: ModelParameters3D) -> None:
    """Render the final circular surface as a three-dimensional height field.

    :param final_state: Final mutable state returned by the model.
    :param params: Model parameters controlling grid spacing.
    :return: None. A Matplotlib figure is displayed.
    """
    x_grid, y_grid = np.meshgrid(final_state["x_mm"], final_state["y_mm"])
    fig = plt.figure(figsize=(10, 7), constrained_layout=True)
    ax = fig.add_subplot(111, projection="3d")
    height_values = masked_for_plot(final_state["height_mm"])
    active_height = active_values(final_state["height_mm"], final_state["active_mask"])
    min_height = float(np.min(active_height))
    max_height = float(np.max(active_height))
    height_span = max_height - min_height
    fallback_span = max(max_height, params.initial_mat_thickness_mm, 1.0)
    z_padding = params.surface_render_z_padding_fraction * max(height_span, fallback_span)

    surface = ax.plot_surface(x_grid, y_grid, height_values, cmap=YLORBR_CMAP, linewidth=0.0, antialiased=True, rstride=1, cstride=1)
    ax.set_title("Final 3-D circular stromatolite surface")
    ax.set_xlabel("x-position (mm)")
    ax.set_ylabel("y-position (mm)")
    ax.set_zlabel("Height above base (mm)")
    ax.set_zlim(max(0.0, min_height - z_padding), max_height + z_padding)
    ax.view_init(elev=32, azim=-135)
    fig.colorbar(surface, ax=ax, shrink=0.62, pad=0.08, label="Surface height (mm)")

    plt.savefig(IMAGE_DIR / "3d-circular-final-surface-render.png", dpi=300)
    plt.show()


In [ ]:
def plot_final_surface_maps(final_state: dict, history: list[dict], params: ModelParameters3D) -> None:
    """Plot final surface height, biomass, EPS, sediment, and light maps.

    :param final_state: Final mutable state returned by the model.
    :param history: List of timestep summary dictionaries from the model run.
    :param params: Model parameters controlling grid spacing.
    :return: None. A Matplotlib figure is displayed.
    """
    extent = [0.0, (params.nx - 1) * params.dx_mm, 0.0, (params.ny - 1) * params.dx_mm]
    panels = [
        ("Final height", final_state["height_mm"], "mm"),
        ("Final biomass", final_state["biomass"], "relative"),
        ("Final EPS", final_state["eps"], "relative"),
        ("Final active sediment", final_state["active_sediment_mm"], "mm"),
        ("Final light at mat", history[-1]["light_grid"], "relative"),
        ("Final layer id", np.where(final_state["active_mask"], final_state["layer_id"], np.nan), "lamina"),
    ]
    fig, axes = plt.subplots(2, 3, figsize=(13, 8), constrained_layout=True)
    cmap = circular_plot_cmap()
    for ax, (title, values, label) in zip(axes.ravel(), panels):
        image = ax.imshow(masked_for_plot(values), origin="lower", extent=extent, cmap=cmap)
        ax.set_title(title)
        ax.set_xlabel("x-position (mm)")
        ax.set_ylabel("y-position (mm)")
        fig.colorbar(image, ax=ax, label=label, shrink=0.85)

    plt.savefig(IMAGE_DIR / "3d-circular-final-surface-maps.png", dpi=300)
    plt.show()


In [ ]:
def plot_cross_section_slice(stratigraphy: dict, final_state: dict, params: ModelParameters3D) -> None:
    """Plot a y-slice through the final laminated 3-D archive.

    :param stratigraphy: Dictionary of stratigraphic raster arrays.
    :param final_state: Final mutable state returned by the model.
    :param params: Model parameters controlling horizontal and vertical plotting extents.
    :return: None. A Matplotlib figure is displayed.
    """
    slice_index = selected_slice_index(params)
    sediment_fraction = stratigraphy["sediment_fraction"][:, slice_index, :]
    deposition = stratigraphy["deposition_increment_mm"][:, slice_index, :]
    cumulative_height = np.cumsum(np.nan_to_num(deposition, nan=0.0), axis=0)
    final_height = final_state["height_mm"][slice_index, :]
    x_mm = final_state["x_mm"]
    ymax = max(float(np.nanmax(cumulative_height)), float(np.nanmax(final_height)))

    fig, ax = plt.subplots(figsize=(12, 6), constrained_layout=True)
    ax.imshow(masked_for_plot(sediment_fraction), aspect="auto", origin="lower", extent=[x_mm.min(), x_mm.max(), 0.0, ymax], cmap=circular_plot_cmap())
    ax.plot(x_mm, final_height, color=YLORBR_PALETTE["pale"], linewidth=2.5, label="Final smoothed surface")
    ax.set_title(f"Final circular cross-section slice at y = {final_state['y_mm'][slice_index]:.1f} mm")
    ax.set_xlabel("x-position (mm)")
    ax.set_ylabel("Height above base (mm)")
    ax.legend(frameon=False)

    plt.savefig(IMAGE_DIR / "3d-circular-final-cross-section-slice.png", dpi=300)
    plt.show()


In [ ]:
def save_final_surface(final_state: dict, output_path: Path) -> None:
    """Save the final 3-D circular surface state as a CSV table.

    :param final_state: Final mutable state returned by the model.
    :param output_path: Destination CSV path.
    :return: None. The CSV file is written to disk.
    """
    output_path.parent.mkdir(parents=True, exist_ok=True)
    x_grid, y_grid = np.meshgrid(final_state["x_mm"], final_state["y_mm"])
    table = np.column_stack([
        x_grid.ravel(),
        y_grid.ravel(),
        final_state["active_mask"].ravel().astype(int),
        final_state["height_mm"].ravel(),
        final_state["biomass"].ravel(),
        final_state["eps"].ravel(),
        final_state["active_sediment_mm"].ravel(),
        final_state["layer_id"].ravel(),
    ])
    header = "x_mm,y_mm,active,height_mm,biomass,eps,active_sediment_mm,layer_id"
    np.savetxt(output_path, table, delimiter=",", header=header, comments="", fmt="%.6f")


## Run The Simulation

In [ ]:
params = load_3d_parameters(BIOLOGY_CONFIG_FILE, GEOMETRY_CONFIG_FILE)
final_state, history, stratigraphy, snapshots = run_3d_model(params)

print(f"Grid cells: {params.nx} x {params.ny}")
print(f"Active circular cells: {history[-1]['active_cell_count']}")
print(f"Footprint radius: {params.footprint_radius_mm or 0.5 * min((params.nx - 1) * params.dx_mm, (params.ny - 1) * params.dx_mm):.2f} mm")
print(f"Final mean height: {history[-1]['mean_height_mm']:.2f} mm")
print(f"Final maximum height: {history[-1]['max_height_mm']:.2f} mm")
print(f"Final surface relief: {history[-1]['relief_mm']:.2f} mm")
print(f"Final surface roughness: {history[-1]['surface_roughness_mm']:.2f} mm")
print(f"Total local burial events: {int(stratigraphy['burial_map'].sum())}")


In [ ]:
plot_growth_summary_3d(history, params)

In [ ]:
plot_environmental_forcing_3d(history, params)

In [ ]:
plot_surface_snapshots_3d(snapshots, params)

In [ ]:
plot_final_surface_render(final_state, params)

In [ ]:
plot_final_surface_maps(final_state, history, params)

In [ ]:
plot_cross_section_slice(stratigraphy, final_state, params)

## Save The 3-D Output

In [ ]:
surface_output_file = OUTPUT_DIR / "3d-circular-final-surface.csv"
stratigraphy_output_file = OUTPUT_DIR / "3d-circular-stratigraphy.npz"

save_final_surface(final_state, surface_output_file)
save_stratigraphy_rasters(stratigraphy, stratigraphy_output_file)

print(f"Saved final surface to {surface_output_file}")
print(f"Saved stratigraphy rasters to {stratigraphy_output_file}")
